In [1]:
import os
from dotenv import load_dotenv

from langchain_groq import ChatGroq

c:\Users\Ali Akbar\anaconda3\envs\ml\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Ali Akbar\anaconda3\envs\ml\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
load_dotenv()  # Load environment variables from .env file

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.8,
    api_key=os.getenv("GROQ_API_KEY")
)

llm.invoke("What is the capital of France?").content

'The capital of France is Paris.'

### load csv files using CSVLoader()

In [ ]:
from langchain_community.document_loaders.csv_loader import CSVLoader

loader = CSVLoader("faqs.csv", source_column="prompt")
data = loader.load()

### embedding of whole csv document

In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"  # Small, fast, free
)

C:\Users\Ali Akbar\AppData\Local\Temp\ipykernel_8848\641180431.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2027.07it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
e = embeddings.embed_query("What coding platform?")
len(e)

768

### vector database using FAISS

In [10]:
from langchain_community.vectorstores import FAISS

vectordb = FAISS.from_documents(data, embeddings)

In [11]:
vectordb = FAISS.from_documents(data, embeddings)
retriever = vectordb.as_retriever()

rdocs = retriever.invoke("How about job placement support?")
rdocs

[Document(id='0da41200-1b51-4417-ae01-5a0d0580b621', metadata={'source': 'Do you provide any job assistance?', 'row': 11}, page_content='prompt: Do you provide any job assistance?\nresponse: Yes, We help you with resume and interview preparation along with that we help you in building online credibility, and based on requirements we refer candidates to potential recruiters.'),
 Document(id='de1f6f49-c773-4251-b771-44b8206cd751', metadata={'source': 'Will this course guarantee me a job?', 'row': 33}, page_content='prompt: Will this course guarantee me a job?\nresponse: We created a much lighter version of this course on YouTube available for free (click this link) and many people gave us feedback that they were able to fetch jobs (see testimonials). Now this paid course is at least 5x better than the YouTube course which gives us ample confidence that you will be able to get a job. However, we want to be honest and do not want to make any impractical promises! Our guarantee is to prepar

In [12]:
for doc in rdocs:
    print(doc.page_content)

prompt: Do you provide any job assistance?
response: Yes, We help you with resume and interview preparation along with that we help you in building online credibility, and based on requirements we refer candidates to potential recruiters.
prompt: Will this course guarantee me a job?
response: We created a much lighter version of this course on YouTube available for free (click this link) and many people gave us feedback that they were able to fetch jobs (see testimonials). Now this paid course is at least 5x better than the YouTube course which gives us ample confidence that you will be able to get a job. However, we want to be honest and do not want to make any impractical promises! Our guarantee is to prepare you for the job market by teaching the most relevant skills, knowledge & timeless principles good enough to fetch the job.
prompt: How can I get help if I have a doubt and need support?
response: We have an active discord server where you can post your question and most of the t

In [13]:
# top k similarity score

docs_and_scores = vectordb.similarity_search_with_score(
    "how about job placement support?",
    k=3
)
for doc, score in docs_and_scores:
    print("Score:", score)
    print("Content:", doc.page_content)
    print("-" * 50)

Score: 1.0289168
Content: prompt: Do you provide any job assistance?
response: Yes, We help you with resume and interview preparation along with that we help you in building online credibility, and based on requirements we refer candidates to potential recruiters.
--------------------------------------------------
Score: 1.2964703
Content: prompt: Will this course guarantee me a job?
response: We created a much lighter version of this course on YouTube available for free (click this link) and many people gave us feedback that they were able to fetch jobs (see testimonials). Now this paid course is at least 5x better than the YouTube course which gives us ample confidence that you will be able to get a job. However, we want to be honest and do not want to make any impractical promises! Our guarantee is to prepare you for the job market by teaching the most relevant skills, knowledge & timeless principles good enough to fetch the job.
--------------------------------------------------
Sc

In [19]:
rdocs[0].page_content

'prompt: Do you provide any job assistance?\nresponse: Yes, We help you with resume and interview preparation along with that we help you in building online credibility, and based on requirements we refer candidates to potential recruiters.'

### final retrieval answers

In [21]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [20]:
def format_docs(docs):
            return "\n\n".join([doc.page_content for doc in docs])

In [34]:
prompt = ChatPromptTemplate.from_template(
        """Given the following context and a question, generate an answer based on this context only.
In the answer try to provide as much text as possible from "response" section in the source document context without making much changes.
If the answer is not found in the context, kindly state "I don't know." Don't try to make up an answer.

CONTEXT: {context}

QUESTION: {question} """
    )

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [40]:
query = "Why this course is exceptional??"

retrieved_docs = retriever.invoke(query)
answer = rag_chain.invoke(query)
answer

'Most of the courses available on the internet teach you how to build x & y without any business context and do not prepare you for real business world problem-solving. This course is rather an experience in which you will learn Excel by solving real-life use cases in an imaginary company called AtliQ Hardware. The tutorials are very easy to understand and also have a lot of fun elements into them so that you don’t get bored. Additionally, the course covers concepts outside the tools such as business context, problem solving, and project management tools, which can be beneficial for students. The course also provides guidance on how to add the learnings from the course to a resume to appeal to hiring teams. Furthermore, the instructor, Dhaval Patel, has a popular data science YouTube channel called Codebasics, where one can watch his videos and read comments to get an idea of his teaching style. Many videos in the course are free, allowing potential students to get an idea of the quali

In [41]:
retrieved_docs

[Document(id='6a39324e-efa3-4eba-a8a3-064821fed4d6', metadata={'source': 'What is different in this course compared to hundreds of courses on the internet and free tutorials on YouTube?', 'row': 18}, page_content='prompt: What is different in this course compared to hundreds of courses on the internet and free tutorials on YouTube?\nresponse: Most of the courses available on the internet teach you how to build x & y without any business context and do not prepare you for real business world problem-solving. This course is rather an experience in which you will learn Excel by solving real-life use cases in an imaginary company called AtliQ Hardware. The tutorials are very easy to understand and also have a lot of fun elements into them so that you don’t get bored ??'),
 Document(id='f9bfad25-15cc-4f13-bd7f-20a70bf1fdca', metadata={'source': 'I use tableau, can I take this course?', 'row': 28}, page_content='prompt: I use tableau, can I take this course?\nresponse: Yes, you will still be

In [44]:
rdocs[0].page_content

'prompt: Do you provide any job assistance?\nresponse: Yes, We help you with resume and interview preparation along with that we help you in building online credibility, and based on requirements we refer candidates to potential recruiters.'

In [45]:
for doc in rdocs:
    print(doc.page_content)

prompt: Do you provide any job assistance?
response: Yes, We help you with resume and interview preparation along with that we help you in building online credibility, and based on requirements we refer candidates to potential recruiters.
prompt: Will this course guarantee me a job?
response: We created a much lighter version of this course on YouTube available for free (click this link) and many people gave us feedback that they were able to fetch jobs (see testimonials). Now this paid course is at least 5x better than the YouTube course which gives us ample confidence that you will be able to get a job. However, we want to be honest and do not want to make any impractical promises! Our guarantee is to prepare you for the job market by teaching the most relevant skills, knowledge & timeless principles good enough to fetch the job.
prompt: How can I get help if I have a doubt and need support?
response: We have an active discord server where you can post your question and most of the t